# Tutorial: Adicionando um novo tribunal
Este notebook demonstra como criar e registrar um adaptador.

In [ ]:
from datetime import date

from models.diario import Diario
from models.interfaces import DiarioAnalyzer, DiarioDiscovery, DiarioDownloader, TribunalAdapter
from tribunais import get_adapter, register_tribunal

## Passo 1: Defina classes básicas

In [ ]:
class DummyDiscovery(DiarioDiscovery):
    def get_diario_url(self, target_date: date) -> str:
        return f"https://example.com/{target_date.isoformat()}.pdf"

    def get_latest_diario_url(self):
        return self.get_diario_url(datetime.now(timezone.utc).date())

    @property
    def tribunal_code(self) -> str:
        return "dummy"


class DummyDownloader(DiarioDownloader):
    def download_diario(self, diario: Diario) -> Diario:
        diario.update_status("downloaded")
        return diario

    def archive_to_ia(self, diario: Diario) -> Diario:
        diario.ia_identifier = "ia-dummy"
        return diario


class DummyAnalyzer(DiarioAnalyzer):
    def extract_decisions(self, diario: Diario):
        return []


class DummyAdapter(TribunalAdapter):
    @property
    def discovery(self):
        return DummyDiscovery()

    @property
    def downloader(self):
        return DummyDownloader()

    @property
    def analyzer(self):
        return DummyAnalyzer()

    @property
    def tribunal_code(self) -> str:
        return "dummy"

## Passo 2: Registrar e usar o adaptador

In [ ]:
register_tribunal("dummy", DummyDiscovery, DummyDownloader, DummyAnalyzer, DummyAdapter)
adapter = get_adapter("dummy")
diario = adapter.create_diario(date(2025, 1, 1))